# 01. Data Quality Audit & Clean Dataset Export

Comprehensive quality audit on raw FortyGuard + Open-Meteo dataset.

In [7]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))

import pandas as pd
import numpy as np
from src.processing.data_quality import load_raw_data, audit_data_quality, clean_and_validate_data

In [8]:
# 1. Load Raw Dataset
df_raw = load_raw_data('../data/raw/temperature_2024_2026.csv')
print(f'Raw Dataset Shape: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns')

Raw Dataset Shape: 369,054 rows, 19 columns


In [9]:
# 2. Run Comprehensive Data Quality Audit
audit = audit_data_quality(df_raw)

print('=== DATA QUALITY SUMMARY ===')
print(f'Total Rows: {audit["shape"][0]:,}, Columns: {audit["shape"][1]}')
print(f'Total Missing Values: {audit["total_missing"]}')
print(f'Exact Duplicate Rows: {audit["exact_duplicates"]}')
print(f'Duplicate (tile_id, timestamp) pairs: {audit["tile_time_duplicates"]}')
print(f'Unique Tiles: {audit["unique_tiles"]:,}, Unique Coordinates: {audit["unique_coords"]:,}')
print(f'Date Range: {audit["date_range"][0]} to {audit["date_range"][1]} ({audit["unique_timestamps"]} days)')
print(f'Daily Frequency Confirmed: {audit["is_daily_frequency"]}')
print(f'Coordinates Valid (within WGS84 bounds): {audit["coordinates_valid"]}')

=== DATA QUALITY SUMMARY ===
Total Rows: 369,054, Columns: 19
Total Missing Values: 0
Exact Duplicate Rows: 0
Duplicate (tile_id, timestamp) pairs: 0
Unique Tiles: 2,187, Unique Coordinates: 3,960
Date Range: 2024-06-01 to 2026-07-30 (182 days)
Daily Frequency Confirmed: True
Coordinates Valid (within WGS84 bounds): True


In [10]:
# 3. Temperature Statistics & Physical Consistency
print('=== TEMPERATURE METRICS ===')
for k, v in audit['temperature_stats'].items():
    print(f'  {k}: {v}')

print('\n=== THERMODYNAMIC & PHYSICAL BOUND CHECKS ===')
for check, passed in audit['physical_checks'].items():
    print(f'  {check}: {"PASSED" if passed else "FAILED"}')

=== TEMPERATURE METRICS ===
  mean: 29.69263367962413
  std: 1.2483032055395145
  min: 25.1142
  25%: 28.8417
  50%: 29.7954
  75%: 30.7046
  max: 32.6247
  outliers_3sigma: 1764

=== THERMODYNAMIC & PHYSICAL BOUND CHECKS ===
  temp_2m_order_valid: PASSED
  apparent_temp_order_valid: PASSED
  wind_gust_order_valid: PASSED
  non_negative_precipitation: PASSED
  non_negative_radiation: PASSED


In [5]:
# 4. Save Validated Dataset
df_cleaned = clean_and_validate_data(df_raw, output_path='data/processed/cleaned_data.csv')
print(f'Cleaned dataset successfully saved to data/processed/cleaned_data.csv ({df_cleaned.shape[0]:,} rows).')

Cleaned dataset successfully saved to data/processed/cleaned_data.csv (369,054 rows).
